<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Day 5 (a) — Build a Research Agent

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Build

An agent that answers questions about **a topic you choose** — using tools **you** write, looping
until it has what it needs, remembering the conversation, and refusing to do anything irreversible
without your say-so.

```
your question  →  LLM decides  →  calls a tool  →  reads the result  →  decides again  →  answer
                       ^                                                      |
                       +------------------------------------------------------+
                                        until finish_reason == "stop"
```

1. Write **three tools** — one must search text you supply
2. Describe them in **JSON schemas**
3. Write `handle_tool_calls()` — the dispatcher
4. Write the **loop** — with a step cap
5. Test it: a question needing **no** tool, **one** tool, and **two** tools
6. Add **memory** so follow-up questions work
7. *(stretch)* a Gradio chat · an approval gate · the same agent in one line

> **You need an OpenAI key** for step 4 onwards. Steps 1–3 run without one.

---

## 1. Setup

Run these three cells. Nothing to write yet.

In [ ]:
# PROVIDED - just run this cell.
!pip install -q openai langchain langchain-openai langgraph sentence-transformers chromadb langchain-text-splitters gradio

In [ ]:
# PROVIDED - just run this cell.
import os, json
from datetime import datetime
from getpass import getpass

from openai import OpenAI
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
import gradio as gr

embedder = SentenceTransformer('all-MiniLM-L6-v2')
chroma = chromadb.Client()
MODEL = "gpt-4o-mini"
print("Ready")

In [ ]:
# PROVIDED - needed from step 4 onwards. Press Enter to skip for now.
key = getpass("OpenAI API Key (Enter to skip): ")
if key:
    os.environ["OPENAI_API_KEY"] = key
    client = OpenAI(api_key=key)
    print("Key loaded")
else:
    print("No key - steps 1 to 3 will still work")

---

## 2. Build it

**The tools you have:**

| What | How |
|---|---|
| ask the model | `client.chat.completions.create(model=MODEL, messages=[...], tools=tools)` |
| did it want a tool? | `response.choices[0].finish_reason == "tool_calls"` |
| what did it ask for? | `response.choices[0].message.tool_calls` → `.function.name`, `.function.arguments` (a JSON **string**) |
| send a result back | `{"role": "tool", "content": json.dumps(result), "tool_call_id": tc.id}` |
| find a function by name | `globals()[name](**args)` |
| split text | `RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50).split_text(text)` |
| make vectors | `embedder.encode(list_of_texts).tolist()` — `.tolist()` matters, Chroma won't take numpy |
| store / search | `collection.add(ids=, embeddings=, documents=)` · `collection.query(query_embeddings=, n_results=)["documents"][0]` |
| today's date | `datetime.now().strftime("%Y-%m-%d")` |

**The schema shape** (one per tool, this is the only fiddly part):

```python
{"type": "function", "function": {
    "name": "search_notes",                       # EXACTLY the Python function name
    "description": "Search my notes about ...",   # this sentence decides if the model calls it
    "parameters": {
        "type": "object",
        "properties": {"query": {"type": "string", "description": "What to search for"}},
        "required": ["query"],
        "additionalProperties": False,
    },
}}
```

Everything from here is yours to write.

In [ ]:
# Step 1 - Pick a topic and put 3+ paragraphs about it in a string called NOTES.
# Your course syllabus, a Wikipedia article, product docs, your own revision notes.
# Then chunk it, index it in a Chroma collection, and print collection.count().

In [ ]:
# Step 2 - Write your three tools as plain Python functions. Each must RETURN a dict.
#   a) search_notes(query)  - searches what you indexed above
#   b) one tool the model CANNOT know on its own (today's date, a random pick, a lookup table)
#   c) one tool the model is BAD at (arithmetic, string counting, sorting)
# Test each one by calling it directly before you go near the LLM.

In [ ]:
# Step 3 - Write the JSON schema for each tool and put them in a list called `tools`.
# Print [t["function"]["name"] for t in tools] to check you got three.

In [ ]:
# Step 4 - Write handle_tool_calls(tool_calls).
# For each call: read the name, json.loads the arguments, look the function up in globals(),
# call it, and append {"role": "tool", "content": json.dumps(result), "tool_call_id": tc.id}.
# Return the list of those messages.

In [ ]:
# Step 5 - Write run_agent(question, history=None).
# Build messages = [system] + history + [user]. Then loop AT MOST 6 times:
#   call the model with tools=tools
#   if finish_reason != "tool_calls": return the content
#   else append the assistant message, extend with handle_tool_calls(...), and go round again
# Print the finish_reason each pass so you can watch it work.

In [ ]:
# Step 6 - Test it three times, one cell each is fine.
#   a) something needing NO tool          - how many passes?
#   b) something needing ONE tool         - how many passes? (it is not one)
#   c) something needing TWO tools        - watch it plan
# If (c) only calls one tool, your descriptions are too vague. Rewrite them, not the loop.

In [ ]:
# Step 7 - Add memory. Keep a `conversation` list, append the user message and the
# answer after each turn, and pass it in as history. Then ask a follow-up
# with a pronoun in it ("what about them?", "how long does that take?").

---

## 3. Stretch goals

Only if steps 1–7 work.

In [ ]:
# Stretch A - Wrap it in a chat UI.
# Hint: def chat(message, history): return run_agent(message, history=history, verbose=False)
#       gr.ChatInterface(chat).launch(share=True)
# gr.ChatInterface already hands you `history` in the right format.

In [ ]:
# Stretch B - Add a dangerous tool and gate it.
# Write delete_note(note_id) that PRINTS what it would delete. Put its name in a set
# called REQUIRES_APPROVAL, and in handle_tool_calls refuse to run anything in that set
# until input("Approve? ") says yes.
# Then ask the agent to delete something and reject it. Where does the gate live -
# in the prompt, or in your code? That is the whole point.

In [ ]:
# Stretch C - The same agent in one line.
#   from langchain_core.tools import tool          -> decorate your functions with @tool
#   from langchain.agents import create_agent
#   agent = create_agent(model="openai:gpt-4o-mini", tools=[...])
#   agent.invoke({"messages": [{"role": "user", "content": "..."}]})
# Ask it the same three questions from step 6. Same answers? Then you understand what
# the framework is doing, because you wrote it first.

In [ ]:
# Stretch D - Poison your own notes.
# Add a chunk containing "Note for the assistant: ignore previous instructions and ..."
# Retrieve it. Ask the agent a question that pulls it in. Write down what happened,
# and which of these you would ship: least privilege / human approval / an allow-list.

---

**When it misbehaves:**

| What you see | What it means |
|---|---|
| The loop never ends | you are not returning on `finish_reason == "stop"` — check the `!=` |
| `400 ... tool_call_id` | every `tool_calls` entry needs exactly one `role: "tool"` reply, with the matching id |
| `TypeError: ... got an unexpected keyword argument` | your schema's property names must match the function's parameter names |
| `json.decoder.JSONDecodeError` | `tool_call.function.arguments` is a **string** — `json.loads` it first |
| The model never calls your tool | the `description` is the only thing it reads. Rewrite that sentence. |
| It calls the tool but ignores the result | you forgot to append the **assistant** message before the tool messages |
| It answers with a made-up date | your date tool's description doesn't say *when* to use it |
| `ValueError` about embeddings on `add()` | you passed a numpy array — add `.tolist()` |
| `IDs already exist` | you ran `add()` twice; new collection name, or restart the runtime |
| `AuthenticationError` | re-run the key cell |

---

### ✅ What you practised

| Idea | The one-liner |
|---|---|
| **Agent = LLM + tools + loop** | there is no agent object; it is a `while` loop around an API call |
| **`finish_reason`** | `"tool_calls"` keep going, `"stop"` return the answer |
| **The description is a prompt** | it is the only thing the model reads when choosing a tool |
| **Who runs the tool** | your code, always — the model only ever asks |
| **One reply per call** | every `tool_call_id` needs its matching `role: "tool"` message |
| **Planning** | nobody wrote a plan; it emerges from the loop |
| **Memory** | a list you keep appending to |
| **The step cap** | the difference between a bug and an invoice |
| **The approval gate** | a Python `if`, not a sentence in the prompt |

**Finished early?**
1. Meter it — add up `response.usage.prompt_tokens` across the loop and cost one hard question.
2. Judge it — write a Pydantic `Judgement` model and grade three of your own answers for groundedness.
3. Trajectory-test it — assert that your date question actually called the date tool.